<a href="https://colab.research.google.com/github/veronicamarenco/forecast-confidence-luxury/blob/main/Hermes_Inventory_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Hermes Inventory Simulator (Leather Goods)
Author: Veronica Marenco (GitHub project: forecast-confidence-luxury)

An advanced statistical simulation modeling Hermès Leather Goods inventory and sales
over a 3-year (36-month) forecast period.

Features:
- Compounding Year-over-Year (YoY) growth rates per bag model.
- Realistic seasonal demand curves (holiday peaks, summer buying cycles).
- Supply-side constraint logic mimicking elite luxury operations (high sell-through target rates).
- High-fidelity visualization generator using Matplotlib.
- Clean CSV exporting pipeline.
"""

import os
import argparse
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hermès signature color scheme
HERMES_ORANGE = "#F37021"
HERMES_CHARCOAL = "#2D2926"
HERMES_GOLD = "#C5A059"
PLOT_BG = "#FAFAFA"

# Default luxury product profile parameters
PRODUCT_PROFILES = {
    "Birkin": {
        "base_sold": 150,
        "base_available": 180,
        "yoy_growth": 0.035,           # ~3.5% (Extremely tight hand-crafted scarcity)
        "sell_through_target": 0.85,    # Target STR (85% - rarely rests on shelves)
        "seasonality": [0.95, 0.90, 1.00, 1.05, 1.10, 1.25, 0.95, 0.85, 1.00, 1.05, 1.15, 1.45] # Monthly multipliers
    },
    "Kelly": {
        "base_sold": 120,
        "base_available": 140,
        "yoy_growth": 0.030,           # ~3.0% (Inelastic supply constraints)
        "sell_through_target": 0.88,    # Highly constrained allocation
        "seasonality": [0.90, 0.85, 0.95, 1.00, 1.05, 1.20, 0.90, 0.80, 0.95, 1.00, 1.10, 1.40]
    },
    "Evelyne": {
        "base_sold": 350,
        "base_available": 450,
        "yoy_growth": 0.055,           # ~5.5% (Slightly more accessible, higher capacity)
        "sell_through_target": 0.76,    # Standard luxury buffer
        "seasonality": [0.85, 0.80, 0.95, 1.05, 1.15, 1.25, 0.95, 0.85, 1.00, 1.10, 1.20, 1.55]
    },
    "Videpoche": {
        "base_sold": 180,
        "base_available": 240,
        "yoy_growth": 0.060,           # ~6.0% (Trending design, rising popularity)
        "sell_through_target": 0.78,
        "seasonality": [0.80, 0.85, 0.95, 1.00, 1.10, 1.20, 1.00, 0.90, 1.05, 1.10, 1.25, 1.50]
    },
    "Della Cavalleria": {
        "base_sold": 140,
        "base_available": 190,
        "yoy_growth": 0.050,           # ~5.0% (Equestrian heritage classic crossbody)
        "sell_through_target": 0.75,
        "seasonality": [0.85, 0.80, 0.90, 1.00, 1.05, 1.15, 0.95, 0.85, 1.00, 1.05, 1.20, 1.45]
    }
}


class HermesInventorySimulator:
    """
    Simulates inventory metrics (Stock Available, Stock Sold, Ending Inventory, STR)
    for luxury retail analysis.
    """
    def __init__(self, start_year: int = 2024, duration_months: int = 36, random_seed: int = 42):
        self.start_year = start_year
        self.duration_months = duration_months
        self.random_seed = random_seed
        self.timeline = pd.date_range(
            start=f"{start_year}-01-01",
            periods=duration_months,
            freq="MS"
        )
        np.random.seed(self.random_seed)

    def run_simulation(self) -> pd.DataFrame:
        """
        Runs simulation loop using seasonality, growth rates, and luxury allocations.
        """
        all_records = []

        for product, config in PRODUCT_PROFILES.items():
            for step, current_date in enumerate(self.timeline):
                year_idx = step // 12
                month_idx = step % 12

                # Compounding annual YoY growth
                growth_multiplier = (1 + config["yoy_growth"]) ** year_idx

                # Apply seasonal variation
                season_mult = config["seasonality"][month_idx]

                # Simulate demand noise (~3% variation)
                noise = np.random.uniform(0.97, 1.03)

                # Calculate estimated/target units sold
                units_sold = int(config["base_sold"] * growth_multiplier * season_mult * noise)

                # Target luxury Sell-Through Rate (restricting available inventory to ensure high demand)
                target_str = config["sell_through_target"] + np.random.uniform(-0.02, 0.02)
                target_str = max(0.50, min(0.95, target_str)) # bounds

                # Compute available stock dynamically
                units_available = int(units_sold / target_str)

                # Edge-case safety constraint
                if units_available <= units_sold:
                    units_available = units_sold + np.random.randint(5, 15)

                ending_inventory = units_available - units_sold
                sell_through_rate = round((units_sold / units_available) * 100, 1)

                all_records.append({
                    "Date": current_date.strftime("%Y-%m"),
                    "Product Line": "Leather Goods",
                    "Product": product,
                    "Stock Available": units_available,
                    "Stock Sold": units_sold,
                    "Ending Inventory": ending_inventory,
                    "Sell-Through Rate (%)": sell_through_rate
                })

        return pd.DataFrame(all_records)


class HermesVisualizer:
    """
    Handles plotting and visualization pipelines for the simulated luxury outputs.
    """
    @staticmethod
    def generate_kpi_charts(df: pd.DataFrame, output_dir: str = "visualizations"):
        """
        Generates clean, publishable charts for simulated products.
        """
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Style configurations
        plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['figure.facecolor'] = PLOT_BG

        products = df["Product"].unique()

        # Plot 1: Standard Supply vs Demand trajectories for Birkin & Kelly
        fig, axes = plt.subplots(len(products), 1, figsize=(14, 18), sharex=True)
        fig.suptitle("Hermes Inventory Dynamics & Forecast (3-Year Timeline)", fontsize=16, fontweight='bold', color=HERMES_CHARCOAL)

        for i, product in enumerate(products):
            sub_df = df[df["Product"] == product]
            ax = axes[i]

            # Draw timelines
            ax.plot(sub_df["Date"], sub_df["Stock Available"], label="Available Stock (Supply Buffer)", color=HERMES_GOLD, linewidth=2, marker='o', alpha=0.85)
            ax.plot(sub_df["Date"], sub_df["Stock Sold"], label="Stock Sold (Allocated Demand)", color=HERMES_ORANGE, linewidth=2.5, marker='s')

            ax.set_title(f"Product Forecast: {product}", fontsize=13, fontweight='semibold', color=HERMES_CHARCOAL)
            ax.set_ylabel("Units", fontsize=11)
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, linestyle="--", alpha=0.5)
            ax.set_facecolor("#FFFFFF")

            if i == 0:
                ax.legend(loc="upper left", frameon=True, facecolor=PLOT_BG)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        chart_path = os.path.join(output_dir, "hermes_inventory_simulation_trends.png")
        plt.savefig(chart_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
        plt.close()
        print(f"[*] Visualizations saved successfully to: {chart_path}")


def main():
    """
    Main function setting up argparse and driving simulator.
    """
    parser = argparse.ArgumentParser(description="Hermes Inventory Simulator & Forecaster Engine")
    parser.add_argument("--start_year", type=int, default=2024, help="Simulation initiation timeline year (default: 2024)")
    parser.add_argument("--months", type=int, default=36, help="Simulation temporal span in months (default: 36)")
    parser.add_argument("--seed", type=int, default=101, help="Random seed setting for reproducibility")
    parser.add_argument("--csv_out", type=str, default="hermes_leather_goods_3year_forecast.csv", help="Filename of exported CSV")
    parser.add_argument("--viz_dir", type=str, default="visualizations", help="Output directory for visual assets")
    parser.add_argument("--skip_plots", action="store_true", help="Disables visualization generation stage")

    args = parser.parse_args()

    print("[+] Initializing Hermes Inventory Simulator...")
    simulator = HermesInventorySimulator(
        start_year=args.start_year,
        duration_months=args.months,
        random_seed=args.seed
    )

    print(f"[+] Simulating metrics over {args.months} months...")
    simulation_results = simulator.run_simulation()

    # Save to CSV
    simulation_results.to_csv(args.csv_out, index=False)
    print(f"[*] Database compiled. Exported {len(simulation_results)} rows to: '{args.csv_out}'")

    # Display dynamic summary diagnostics
    print("\n--- SIMULATION SUMMARY STATISTICS (3-YEAR TOTALS) ---")
    summary = simulation_results.groupby("Product").agg(
        Total_Sold=pd.NamedAgg(column="Stock Sold", aggfunc="sum"),
        Peak_Available=pd.NamedAgg(column="Stock Available", aggfunc="max"),
        Average_STR=pd.NamedAgg(column="Sell-Through Rate (%)", aggfunc="mean")
    )
    print(summary.to_string())
    print("----------------------------------------------------\n")

    # Plot pipeline
    if not args.skip_plots:
        print("[+] Initiating plotting pipeline...")
        HermesVisualizer.generate_kpi_charts(simulation_results, args.viz_dir)

    print("[+] Task complete.")

if __name__ == "__main__":
    main()

eof

### Why this structure works perfectly for a GitHub project:
1. **Separation of Concerns:** Distinct, OOP-oriented components handle simulation (`HermesInventorySimulator`), plotting (`HermesVisualizer`), and scripting CLI execution parameters (`main()`).
2. **Highly Customisable (CLI Args):** Users checking out your GitHub repo can execute the code with customized parameters directly from their terminal:
   ```bash
   python hermes_simulator.py --months 48 --seed 42 --csv_out custom_hermes_run.csv
   ```
3. **No Brittle Dependencies:** Matplotlib is isolated. It auto-detects system styles gracefully and exports high-definition vectors (`dpi=300`), ready to be uploaded straight onto your portfolio readmes.
  ```